## Basics of Converting an LLM to an Agent from Scratch

- An agentic ai is fundamentally an LLM that can execute an other application ('tool')
- On its own an LLM is static application,a high-dimensional matrix of numbers (weights), that statistically predicts the next word/token. 
- Therefore an LLM cannot hold state or act without prompted
- An agent is that same model wrapped in a **harness** (a scaffolding application) that lets it call external tools (functions, APIs, code execution, database search) and feed the results back in so the LLM can then reason over the tool's output to generate a response back to the user/client.
- `Agent Loop` :
    - the defining feature of agents is the *loop* wherein the model decides an action, the harness executes it and the output is appended to context (memory state), and the model decides again, repeating until the task is done

-  I will use huggingface `TeichAI/Qwen3-32B-Kimi-K2-Thinking-Distill:featherless-ai` that has inference provide 

In [5]:
#get my hugging face api
import os
HF_TOKEN = os.environ["HF_TOKEN"]

In [ ]:
#initialize the client to connect to hugging face

from huggingface_hub import InferenceClient 

In [32]:
#example of LLM with no tools

#choose a HF model that is an Inference Provider label 
client =  InferenceClient(
    token=HF_TOKEN,
    model="TeichAI/Qwen3-32B-Kimi-K2-Thinking-Distill:featherless-ai"
    )


response= client.chat.completions.create(
    messages=[
        {"role":"user", "content":"what is the capital of the Caribbean?"}
        ]
)

response

ChatCompletionOutput(choices=[ChatCompletionOutputComplete(finish_reason='stop', index=0, message=ChatCompletionOutputMessage(role='assistant', content=" The Caribbean is a region comprising many independent countries, territories, and islands, so it doesn't have a single capital city. However, **Georgetown, Guyana** serves as the **administrative capital** of the Caribbean region as the permanent seat of the **Caribbean Community (CARICOM)** secretariat—the main regional organization.\n\nIf you were asking about a specific Caribbean country, I'd be happy to help with that!", reasoning='\nThe user asks: "what is the capital of the Caribbean?" This is a bit ambiguous because the Caribbean is a region, not a single country. It includes many countries and territories. I need to interpret what they might mean.\n\n1. **Caribbean as a region**: The Caribbean is a region comprising numerous countries and dependencies. There is no single "capital" of the entire Caribbean region in the traditio

In [33]:
response.choices[0].message.content

" The Caribbean is a region comprising many independent countries, territories, and islands, so it doesn't have a single capital city. However, **Georgetown, Guyana** serves as the **administrative capital** of the Caribbean region as the permanent seat of the **Caribbean Community (CARICOM)** secretariat—the main regional organization.\n\nIf you were asking about a specific Caribbean country, I'd be happy to help with that!"

In [34]:
response.choices[0].message.__dict__

{'role': 'assistant',
 'content': " The Caribbean is a region comprising many independent countries, territories, and islands, so it doesn't have a single capital city. However, **Georgetown, Guyana** serves as the **administrative capital** of the Caribbean region as the permanent seat of the **Caribbean Community (CARICOM)** secretariat—the main regional organization.\n\nIf you were asking about a specific Caribbean country, I'd be happy to help with that!",
 'reasoning': '\nThe user asks: "what is the capital of the Caribbean?" This is a bit ambiguous because the Caribbean is a region, not a single country. It includes many countries and territories. I need to interpret what they might mean.\n\n1. **Caribbean as a region**: The Caribbean is a region comprising numerous countries and dependencies. There is no single "capital" of the entire Caribbean region in the traditional sense. However, there might be a de facto capital for regional organizations or a city that serves as a major 

## Now with Tool Call
- primitively, a `tool` is a function or callable object 
- so let's give the LLM external functionality i.e. 'agency'

In [22]:
def get_nucleic_acid(seq:str)->str:
    """
    This function takes a string of nucleic acid sequence and returns the type of nucleic acid.
    """
    seq = seq.upper()
    if all(base in "ATCG" for base in seq):
        return "DNA"
    elif all(base in "AUCG" for base in seq):
        return "RNA"
    else:
        return "Invalid nucleic acid sequence"



In [ ]:
#tell the LLM which tool it has available using a schema


get_nucleic_acid_schema = {
    "type":"function",
    "function": { "name":"get_nucleic_acid",
                     "description":"Determine the class of nucleid acid from a valid biological sequence",
                     "parameters": {"type":"object", 
                                    "properties":{
                                        "seq": {
                                            "type":"string", 
                                            "description":"the query biological sequence to classify. strip any internal newlines" #tells LLM what to enter, so be precise
                                            }
                                    }
                                    }
                                    },
    "required":["seq"]
                     }




In [25]:
#can use pydantic to more conveniently create schema 

from pydantic import BaseModel, Field


class GetNucleicAcid(BaseModel):
    seq: str= Field(..., description="the query biological sequence to classify. strip any internal newlines")

schema = {
    "type": "function",
    "function": {
        "name": "get_nucleic_acid",
        "description": "Determine the class of nucleid acid from a valid biological sequence",
        "parameters": GetNucleicAcid.model_json_schema()
    }
}
schema


{'type': 'function',
 'function': {'name': 'get_nucleic_acid',
  'description': 'Determine the class of nucleid acid from a valid biological sequence',
  'parameters': {'properties': {'seq': {'description': 'the query biological sequence to classify. strip any internal newlines',
     'title': 'Seq',
     'type': 'string'}},
   'required': ['seq'],
   'title': 'GetNucleicAcid',
   'type': 'object'}}}

In [35]:
#now give our function tool to the  language model 

response_with_tool = client.chat.completions.create(
    messages=[{"role":"user", "content": "what kind of nucleic acid is this GCCCUACCAGCAGA"}],
    tools=[schema],
    tool_choice="auto" #let the model decide when to use the tool 
)

In [ ]:
#lets see how the model used the tool to answer the question
print(response_with_tool.choices[0].message.reasoning)


The user is asking what kind of nucleic acid the sequence "GCCCUACCAGCAGA" is. I need to determine if it's DNA or RNA. 

First, I'll recall the key differences. DNA typically contains the bases A, T, C, and G. RNA contains A, U, C, and G. The presence of "U" (uracil) indicates RNA, while "T" (thymine) indicates DNA. 

Looking at the sequence: GCCCUACCAGCAGA. Let me check each base. G, C, C, C, U, A, C, C, A, G, C, A, G, A. The key base to check is the third position where there's a "U". Since "U" is present, this is RNA. 

I should use the provided function `get_nucleic_acid` and pass the sequence as the argument. The function expects a string parameter "seq". I'll make sure to strip any internal newlines, but in this case, the sequence is already a single line. 

So, the function call should be:
{"name": "get_nucleic_acid", "arguments": {"seq": "GCCCUACCAGCAGA"}}

This should return the result that it's RNA. I'll present that to the user clearly.



In [ ]:
#lets make an Agent so we dont have to manually retrieve the response mssage
import json 


class Agent:
    #to initialize an agent we need an inference client, system prompt and list of tools 
    def __init__(self, client:InferenceClient, system:str="", tools:list=None):
        self.client = client
        self.system = system
        self.messages: list =[]
        self.tools = tools if tools is not None else []
        if self.system:
            self.messages.append({"role":"system", "content":system})

    def __call__(self, message=""):
        self.messages.append({"role":"user", "content":message})

        final_assistant_content = self.execute()

        if final_assistant_content:
            self.messages.append({"role": "assistant", "content":final_assistant_content})

        return final_assistant_content

    def execute(self):
        while True:
            completion = self.client.chat.completions.create(
                messages=self.messages,
                tools=self.tools,
                tool_choice="auto"
            )

            response_message = completion.choices[0].message

            #let's track tool call in the history 
            if response_message.tool_calls:
                self.messages.append({response_message})
                tool_outputs = []

                #loop through the tool calls, in case there are multiple tools
                #for each tool called 
                for tool_call in response_message.tool_calls:
                    function_name = tool_call.function.name
                    function_args = json.loads(tool_call.function.arguments)

                    #execute the tool
                    if function_name in globals() and callable(globals()[function_name]):
                        function_to_call = globals()[function_name]
                        executed_output = function_to_call(**function_args)
                        tool_output_content = str(executed_output)
                        print(f"Executing tool: {function_name} with args {function_args}, Output: {tool_output_content[:500]}...")

                    tool_outputs.append({
                        "tool_call_id": tool_call.id, #essential to record
                        "role":"tool",
                        "name": function_name,
                        "content":tool_output_content
                    })

                self.messages.extend(tool_outputs) #update the message history with the tool outputs
            #if there were no tool return the native LLM generated mesage
            else:
                return  response_message.content


In [40]:
#to initialize an agent we need an inference client, system prompt and list of tools 
client = InferenceClient(
    token=HF_TOKEN,
   model="TeichAI/Qwen3-32B-Kimi-K2-Thinking-Distill:featherless-ai"
)
system_prompt="You are helpful molecular biology assistant who can classify nucleic acids"
tools=[schema]

agent = Agent(client=client, system=system_prompt, tools=tools)

In [41]:
#lets query the agent to classify a nucleic acid sequence
query = "what kind of nucleic acid is this GCCCUACCAGCAGA"
agent(query)

AttributeError: 'Agent' object has no attribute 'message'